In [4]:
import collections
import logging
import os
import pathlib
import re
import string
import sys
import time

import numpy as np
import matplotlib.pyplot as plt

import tensorflow_datasets as tfds
import tensorflow_text as text
import tensorflow as tf

In [11]:
logging.getLogger('tensorflow').setLevel(logging.ERROR)  # suppress warnings

In [12]:
from pathlib import Path

def load_data(path):
    path = Path(path)  # Convert string to Path object
    text = path.read_text(encoding='utf-8')

    lines = text.splitlines()
    pairs = [line.split('\t') for line in lines]

    inp = [inp for targ, inp in pairs]
    targ = [targ for targ, inp in pairs]

    return inp, targ

def load_data_as_dataset(path):
    inp, targ = load_data(path)
    dataset = tf.data.Dataset.from_tensor_slices((inp, targ))
    return dataset

In [13]:
train_examples = load_data_as_dataset('translation_train.txt')

In [14]:
val_examples = load_data_as_dataset('translation_val.txt')


In [15]:
for pt_examples, en_examples in train_examples.batch(3).take(1):
    for pt in pt_examples.numpy():
        print(pt.decode('utf-8'))

    print()

    for en in en_examples.numpy():
        print(en.decode('utf-8'))

Il boit de l’eau.
Il se plaint toujours.
Quoi ? Quelque chose.

A bi ji min na
A le dalakolontɛ lon bɛ.
Mun? Fɛn dɔ.


2024-07-02 14:56:04.131611: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


# Make Tokenizer

In [27]:
for fr, dyu in train_examples.take(1):
    print("Dioula: ", dyu.numpy().decode('utf-8'))
    print("French:   ",  fr.numpy().decode('utf-8'))

Dioula:  A bi ji min na
French:    Il boit de l’eau.


2024-07-02 15:05:49.813407: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [28]:
train_fr = train_examples.map(lambda fr, dyu: fr)
train_dyu = train_examples.map(lambda fr, dyu: dyu)

In [29]:
from tensorflow_text.tools.wordpiece_vocab import bert_vocab_from_dataset as bert_vocab

In [30]:
bert_tokenizer_params=dict(lower_case=True)
reserved_tokens=["[PAD]", "[UNK]", "[START]", "[END]"]

bert_vocab_args = dict(
        # The target vocabulary size
        vocab_size = 8000,
        # Reserved tokens that must be included in the vocabulary
        reserved_tokens=reserved_tokens,
        # Arguments for `text.BertTokenizer`
        bert_tokenizer_params=bert_tokenizer_params,
        # Arguments for `wordpiece_vocab.wordpiece_tokenizer_learner_lib.learn`
        learn_params={},
)

In [31]:
%%time
dyu_vocab = bert_vocab.bert_vocab_from_dataset(
        train_dyu.batch(1000).prefetch(2),
        **bert_vocab_args
)

2024-07-02 15:05:58.948997: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


CPU times: user 3.1 s, sys: 99.8 ms, total: 3.2 s
Wall time: 3.18 s


In [32]:
print(dyu_vocab[:10])
print(dyu_vocab[100:110])
print(dyu_vocab[1000:1010])
print(dyu_vocab[-10:])

['[PAD]', '[UNK]', '[START]', '[END]', '!', "'", ',', '-', '.', '/']
['ke', '##ma', 'sɔrɔ', '##si', 'mɔgɔ', '##le', 'ani', 'kɛra', '##man', 'kalan']
[]
['##0', '##1', '##;', '##?', '##[', '##j', '##q', '##v', '##x', '##ŋ']


In [33]:
def write_vocab_file(filepath, vocab):
    with open(filepath, 'w') as f:
        for token in vocab:
            print(token, file=f)

In [40]:
write_vocab_file('dyu_vocab.txt', dyu_vocab)

In [41]:
%%time
fr_vocab = bert_vocab.bert_vocab_from_dataset(
        train_fr.batch(1000).prefetch(2),
        **bert_vocab_args
)

2024-07-02 15:10:00.014377: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


CPU times: user 5.7 s, sys: 106 ms, total: 5.81 s
Wall time: 5.95 s


In [42]:
print(fr_vocab[:10])
print(fr_vocab[100:110])
print(fr_vocab[1000:1010])
print(fr_vocab[-10:])

['[PAD]', '[UNK]', '[START]', '[END]', '!', '"', '&', "'", '(', ')']
['##es', '##a', 'qui', '##re', 'etait', '##ment', '##l', 'ou', 'suis', 'tout']
['##rain', '##ran', '##red', '##renez', '##rer', '##rles', '##rque', '##sons', '##sy', '##tique']
['##ł', '##œ', '##ʻ', '##α', '##–', '##—', '##’', '##“', '##”', '##€']


In [43]:
write_vocab_file('fr_vocab.txt', fr_vocab)

In [44]:
import glob

txt_files = glob.glob('*.txt')
print(txt_files)

['translation_train.txt', 'dyu_vocab.txt', 'translation.txt', 'translation_val.txt', 'pt_vocab.txt', 'fr_vocab.txt', 'en_vocab.txt', 'translation_test.txt']


In [45]:
dyu_tokenizer = text.BertTokenizer('dyu_vocab.txt', **bert_tokenizer_params)
fr_tokenizer = text.BertTokenizer('fr_vocab.txt', **bert_tokenizer_params)

In [46]:
for fr_examples, dyu_examples in train_examples.batch(3).take(1):
    for ex in fr_examples:
        print(ex.numpy())

b'Il boit de l\xe2\x80\x99eau.'
b'Il se plaint toujours.'
b'Quoi ? Quelque chose.'


2024-07-02 15:12:03.715329: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [47]:
# Tokenize the examples -> (batch, word, word-piece)
token_batch = fr_tokenizer.tokenize(fr_examples)
# Merge the word and word-piece axes -> (batch, tokens)
token_batch = token_batch.merge_dims(-2,-1)

for ex in token_batch.to_list():
    print(ex)

[59, 19, 486, 56, 29, 52, 252, 12]
[59, 88, 33, 299, 298, 302, 12]
[282, 17, 433, 223, 12]


In [48]:
# Lookup each token id in the vocabulary.
txt_tokens = tf.gather(fr_vocab, token_batch)
# Join with spaces.
tf.strings.reduce_join(txt_tokens, separator=' ', axis=-1)

<tf.Tensor: shape=(3,), dtype=string, numpy=
array([b'il b ##oit de l \xe2\x80\x99 eau .',
       b'il se p ##la ##int toujours .', b'quoi ? quelque chose .'],
      dtype=object)>

In [49]:
words = fr_tokenizer.detokenize(token_batch)
tf.strings.reduce_join(words, separator=' ', axis=-1)

<tf.Tensor: shape=(3,), dtype=string, numpy=
array([b'il boit de l \xe2\x80\x99 eau .', b'il se plaint toujours .',
       b'quoi ? quelque chose .'], dtype=object)>

In [50]:
START = tf.argmax(tf.constant(reserved_tokens) == "[START]")
END = tf.argmax(tf.constant(reserved_tokens) == "[END]")

def add_start_end(ragged):
    count = ragged.bounding_shape()[0]
    starts = tf.fill([count,1], START)
    ends = tf.fill([count,1], END)
    return tf.concat([starts, ragged, ends], axis=1)

In [51]:
words = fr_tokenizer.detokenize(add_start_end(token_batch))
tf.strings.reduce_join(words, separator=' ', axis=-1)

<tf.Tensor: shape=(3,), dtype=string, numpy=
array([b'[START] il boit de l \xe2\x80\x99 eau . [END]',
       b'[START] il se plaint toujours . [END]',
       b'[START] quoi ? quelque chose . [END]'], dtype=object)>

In [52]:
def cleanup_text(reserved_tokens, token_txt):
    # Drop the reserved tokens, except for "[UNK]".
    bad_tokens = [re.escape(tok) for tok in reserved_tokens if tok != "[UNK]"]
    bad_token_re = "|".join(bad_tokens)

    bad_cells = tf.strings.regex_full_match(token_txt, bad_token_re)
    result = tf.ragged.boolean_mask(token_txt, ~bad_cells)

    # Join them into strings.
    result = tf.strings.reduce_join(result, separator=' ', axis=-1)

    return result

In [53]:
fr_examples.numpy()

array([b'Il boit de l\xe2\x80\x99eau.', b'Il se plaint toujours.',
       b'Quoi ? Quelque chose.'], dtype=object)

In [56]:
token_batch = fr_tokenizer.tokenize(fr_examples).merge_dims(-2,-1)
words = fr_tokenizer.detokenize(token_batch)
words

<tf.RaggedTensor [[b'il', b'boit', b'de', b'l', b'\xe2\x80\x99', b'eau', b'.'],
 [b'il', b'se', b'plaint', b'toujours', b'.'],
 [b'quoi', b'?', b'quelque', b'chose', b'.']]>

In [57]:
cleanup_text(reserved_tokens, words).numpy()

array([b'il boit de l \xe2\x80\x99 eau .', b'il se plaint toujours .',
       b'quoi ? quelque chose .'], dtype=object)

In [58]:
class CustomTokenizer(tf.Module):
    def __init__(self, reserved_tokens, vocab_path):
        self.tokenizer = text.BertTokenizer(vocab_path, lower_case=True)
        self._reserved_tokens = reserved_tokens
        self._vocab_path = tf.saved_model.Asset(vocab_path)

        vocab = pathlib.Path(vocab_path).read_text().splitlines()
        self.vocab = tf.Variable(vocab)

        ## Create the signatures for export:   

        # Include a tokenize signature for a batch of strings. 
        self.tokenize.get_concrete_function(
                tf.TensorSpec(shape=[None], dtype=tf.string))

        # Include `detokenize` and `lookup` signatures for:
        #   * `Tensors` with shapes [tokens] and [batch, tokens]
        #   * `RaggedTensors` with shape [batch, tokens]
        self.detokenize.get_concrete_function(
                tf.TensorSpec(shape=[None, None], dtype=tf.int64))
        self.detokenize.get_concrete_function(
                tf.RaggedTensorSpec(shape=[None, None], dtype=tf.int64))

        self.lookup.get_concrete_function(
                tf.TensorSpec(shape=[None, None], dtype=tf.int64))
        self.lookup.get_concrete_function(
                tf.RaggedTensorSpec(shape=[None, None], dtype=tf.int64))

        # These `get_*` methods take no arguments
        self.get_vocab_size.get_concrete_function()
        self.get_vocab_path.get_concrete_function()
        self.get_reserved_tokens.get_concrete_function()

    @tf.function
    def tokenize(self, strings):
        enc = self.tokenizer.tokenize(strings)
        # Merge the `word` and `word-piece` axes.
        enc = enc.merge_dims(-2,-1)
        enc = add_start_end(enc)
        return enc

    @tf.function
    def detokenize(self, tokenized):
        words = self.tokenizer.detokenize(tokenized)
        return cleanup_text(self._reserved_tokens, words)

    @tf.function
    def lookup(self, token_ids):
        return tf.gather(self.vocab, token_ids)

    @tf.function
    def get_vocab_size(self):
        return tf.shape(self.vocab)[0]

    @tf.function
    def get_vocab_path(self):
        return self._vocab_path

    @tf.function
    def get_reserved_tokens(self):
        return tf.constant(self._reserved_tokens)

In [59]:
tokenizers = tf.Module()
tokenizers.dyu = CustomTokenizer(reserved_tokens, 'dyu_vocab.txt')
tokenizers.fr = CustomTokenizer(reserved_tokens, 'fr_vocab.txt')

In [60]:
model_name = 'ted_hrlr_translate_dyu_fr_converter'
tf.saved_model.save(tokenizers, model_name)

In [8]:

reloaded_tokenizers = tf.saved_model.load("ted_hrlr_translate_dyu_fr_converter")


In [9]:
reloaded_tokenizers.fr.get_vocab_size().numpy()

1096

In [2]:
model

NameError: name 'model' is not defined